# 1.3 DFL 데드볼 Transformer

- **모델** · 로컬 self-attention 1블록, 4헤드, 임베딩 48, FFN 96, RoPE, 전체 20초 창(앞뒤 10초, 101시점).
- **학습** · 기존 TCN과 같은 선수 특징, AdamW, 학습률 0.002, 가중치 감쇠 0.001, BCE, 최대 20 epoch. attention을 64시점 단위로 묶으므로 한 배치의 학습 시점 수는 TCN보다 작다.
- **검증** · DFL 7경기, 5경기로 학습하고 내부 경기로 epoch 선택 → 선택한 epoch 수로 6경기 재학습 → 남겨둔 경기의 25fps 예측 평가. 대회 경기 미사용.
- **실행** · 같은 폴더의 `dfl_model_comparison.ipynb`와 `dfl_tcn_tuning.ipynb`가 필요하다. 해당 노트북의 Python 환경에서 셀을 순서대로 실행한다.
- **출력** · `model/영상추출/추출csv/dead_ball/dfl_transformer_20261111_01/`.

In [ ]:
import json
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/train_dfl.py").is_file()
)
tuning = root / "model/영상추출/모델/dead_ball/dfl_tcn_tuning.ipynb"
tuning_cells = {cell["id"]: cell for cell in json.loads(tuning.read_text())["cells"]}
for cell_id in ("dfl13-tune-2", "dfl13-tune-4"):
    source = "".join(tuning_cells[cell_id]["source"])
    exec(compile(source, f"{tuning}:{cell_id}", "exec"))

from torch.nn import functional as F

NOTEBOOK = folder / "dfl_transformer.ipynb"
RUN_ID = "dfl_transformer_20261111_01"
OUT = ROOT / "model/영상추출/추출csv/dead_ball" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
GRID = OUT.parent / "dfl_tcn_grid_20261111_01" / "model_summary.csv"
WIDTH = 48
HEADS = 4
FEEDFORWARD = 96
CHUNK = 64
WINDOW_S = 20
RADIUS = WINDOW_S * FPS // (2 * STRIDE)
BATCH_CHUNKS = 16
assert RADIUS == 50 and WIDTH % HEADS == 0 and WIDTH // HEADS % 2 == 0

## 로컬 Transformer

각 중앙 시점의 attention은 같은 하프에서 앞뒤 50개의 샘플을 읽는다. 하프 경계는 기존 TCN처럼 가장자리 특징을 반복해 채운다. 위치 인코딩은 청크 안에서의 상대적 순서를 전달한다.

In [ ]:
class LocalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(len(BASE) * 2, WIDTH)
        self.attention_norm = nn.LayerNorm(WIDTH)
        self.qkv = nn.Linear(WIDTH, WIDTH * 3)
        self.attention_output = nn.Linear(WIDTH, WIDTH)
        self.ffn_norm = nn.LayerNorm(WIDTH)
        self.ffn = nn.Sequential(nn.Linear(WIDTH, FEEDFORWARD), nn.GELU(), nn.Linear(FEEDFORWARD, WIDTH))
        self.classifier = nn.Linear(WIDTH, 1)

        positions = torch.arange(CHUNK + 2 * RADIUS)
        centers = positions[RADIUS:RADIUS + CHUNK]
        mask = (centers[:, None] - positions[None, :]).abs() <= RADIUS
        head_width = WIDTH // HEADS
        frequencies = 10000 ** (-torch.arange(0, head_width, 2, dtype=torch.float32) / head_width)
        phase = positions[:, None] * frequencies[None, :]
        self.register_buffer("mask", mask, persistent=False)
        self.register_buffer("cosine", phase.cos(), persistent=False)
        self.register_buffer("sine", phase.sin(), persistent=False)

    def rotate(self, vector):
        even = vector[..., ::2]
        odd = vector[..., 1::2]
        real = even * self.cosine - odd * self.sine
        imaginary = odd * self.cosine + even * self.sine
        return torch.stack((real, imaginary), dim=-1).flatten(-2)

    def forward(self, values):
        embedded = self.projection(values.transpose(1, 2))
        batch, steps, _ = embedded.shape
        qkv = self.qkv(self.attention_norm(embedded)).reshape(batch, steps, 3, HEADS, WIDTH // HEADS)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        center = slice(RADIUS, RADIUS + CHUNK)
        query = self.rotate(query)[:, :, center].contiguous()
        key = self.rotate(key)
        attention = F.scaled_dot_product_attention(query, key, value.contiguous(), attn_mask=self.mask)
        attention = attention.transpose(1, 2).reshape(batch, CHUNK, WIDTH)
        hidden = embedded[:, center] + self.attention_output(attention)
        hidden = hidden + self.ffn(self.ffn_norm(hidden))
        return self.classifier(hidden).transpose(1, 2)


def smoke_test():
    with torch.no_grad():
        model = LocalTransformer().eval()
        x = torch.randn(2, len(BASE) * 2, CHUNK + 2 * RADIUS)
        logits = model(x)
        assert logits.shape == (2, 1, CHUNK)
        assert model.mask.sum(dim=1).tolist() == [2 * RADIUS + 1] * CHUNK
        x[:, :, RADIUS * 2 + 1:] += 10
        torch.testing.assert_close(logits[:, :, 0], model(x)[:, :, 0], rtol=1e-6, atol=1e-6)
        return sum(parameter.numel() for parameter in model.parameters())


def train_transformer(matches, train, max_epochs, inner=None):
    torch.manual_seed(TORCH_SEED)
    rng = np.random.default_rng(TORCH_SEED)
    train_x, train_y, mean, scale = make_training_data(matches, train, RADIUS)
    model = LocalTransformer()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    best_epoch, best_score, stale = max_epochs, -np.inf, 0
    history = []
    batch_count = (len(train_x) + BATCH_CHUNKS - 1) // BATCH_CHUNKS
    for epoch in range(1, max_epochs + 1):
        model.train()
        for indices in np.array_split(rng.permutation(len(train_x)), batch_count):
            optimizer.zero_grad(set_to_none=True)
            logits = model(train_x[indices]).squeeze(1)
            target = train_y[indices]
            mask = target >= 0
            loss = (criterion(logits, target.clamp_min(0)) * mask).sum() / mask.sum()
            loss.backward()
            optimizer.step()

        if inner is None or epoch < MIN_EPOCHS or epoch % CHECK_EVERY:
            continue
        values, labels = inner[:2]
        period = labels["period"].to_numpy()
        score = predict_tcn(model, values, period, mean, scale, RADIUS)
        state = smoothing.viterbi(score, period, VITERBI["penalty"], VITERBI["bias"])
        current = scoring.score_one(state, labels)["alive_spell_f1"]
        history.append(dict(epoch=epoch, inner_f1=float(current)))
        if current > best_score + 1e-9:
            best_score, best_epoch, stale = current, epoch, 0
        else:
            stale += 1
        if stale >= PATIENCE:
            break

    return (model, mean, scale), best_epoch, history, epoch


def evaluate_fold(held, train, evaluation):
    started = perf_counter()
    inner = MATCHES[(MATCHES.index(held) + 1) % len(MATCHES)]
    selection_matches = [match for match in MATCHES if match not in (held, inner)]
    refit_matches = [match for match in MATCHES if match != held]
    assert len(selection_matches) == 5 and len(refit_matches) == 6
    _, epoch, history, stopped = train_transformer(selection_matches, train, MAX_EPOCHS, evaluation[inner])
    selection_s = perf_counter() - started
    refit_start = perf_counter()
    (model, mean, scale), _, _, _ = train_transformer(refit_matches, train, epoch)
    refit_s = perf_counter() - refit_start
    values, labels, geometry, crossings, spells = evaluation[held]
    score = predict_tcn(model, values, labels["period"].to_numpy(), mean, scale, RADIUS)
    scores = score_states(score, labels, geometry, crossings, spells)
    result = dict(match=held, inner_match=inner, selected_epoch=epoch, inner_stopped_epoch=stopped,
                  selection_s=selection_s, refit_s=refit_s, total_s=perf_counter() - started,
                  history=history, scores=scores)
    print(f"{held} | epoch {epoch}/{stopped} | {result['total_s']:.1f}s | "
          f"F1 {scores[0]['alive_spell_f1']:.4f} / {scores[1]['alive_spell_f1']:.4f}", flush=True)
    return result

## 실행과 비교 결과

시간은 내부 epoch 탐색, 재학습, 검증을 포함한다. 참조 모델의 시간은 기존 실행 기록에서 읽는다.

In [ ]:
def write_reports(results, fingerprint, parameters, preparation_s):
    flattened = []
    histories = []
    for fold in results:
        details = {key: value for key, value in fold.items() if key not in ("scores", "history")}
        flattened.extend(dict(details, **score) for score in fold["scores"])
        histories.extend(dict(match=fold["match"], inner_match=fold["inner_match"], **row)
                         for row in fold["history"])
    scores = pd.DataFrame(flattened)
    assert len(scores) == 14 and set(scores.match) == set(MATCHES)
    assert set(scores.stage) == {"viterbi", "line_out"}
    assert not scores.duplicated(["match", "stage"]).any()
    assert scores.alive_spell_f1.between(0, 1).all() and (scores.match != scores.inner_match).all()

    summary = (scores.groupby("stage", sort=False)
               .agg(alive_spell_f1=("alive_spell_f1", "mean"), dead_frame_f1=("dead_f1", "mean"),
                    false_dead=("false_dead", "sum"), missed_dead=("missed_dead", "sum"),
                    total_s=("total_s", "sum"), selection_s=("selection_s", "sum"), refit_s=("refit_s", "sum"),
                    epoch_min=("selected_epoch", "min"), epoch_max=("selected_epoch", "max")).reset_index())
    summary.insert(0, "model", "Transformer 20s 48 AdamW")
    common = ["stage", "alive_spell_f1", "dead_frame_f1", "false_dead", "missed_dead"]
    references = [
        pd.read_csv(BASELINE_OUTPUT).query("model == 'LightGBM'")[["model", *common, "fold_s"]]
          .rename(columns={"fold_s": "total_s"}),
        pd.read_csv(GRID).query("model == '20s 48 AdamW'")[["model", *common, "total_s"]]
          .assign(model="TCN 20s 48 AdamW"),
    ]
    comparison = pd.concat([*references, summary[["model", *common, "total_s"]]], ignore_index=True)
    scores.to_csv(OUT / "per_match.csv", index=False)
    pd.DataFrame(histories).to_csv(OUT / "inner_epoch_history.csv", index=False)
    summary.to_csv(OUT / "model_summary.csv", index=False)
    comparison.to_csv(OUT / "comparison.csv", index=False)
    files = ("per_match.csv", "inner_epoch_history.csv", "model_summary.csv", "comparison.csv")
    manifest = dict(
        run_id=RUN_ID, fingerprints=fingerprint, matches=MATCHES, architecture=dict(
            parameters=parameters, width=WIDTH, heads=HEADS, blocks=1, feedforward=FEEDFORWARD, window_s=WINDOW_S,
            position_encoding="RoPE", chunk=CHUNK, batch_chunks=BATCH_CHUNKS),
        packages={name: version(name) for name in ("torch", "numpy", "pandas", "scipy", "pyarrow")},
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, torch_seed=TORCH_SEED,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, preparation_s=preparation_s,
        input_feature_sha256=json.loads((BASELINE_OUTPUT.parent / "run_manifest.json").read_text())["input_feature_sha256"],
        output_sha256={name: digest(OUT / name) for name in files},
    )
    (OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (OUT / "validation.json").write_text(json.dumps(dict(passed=True, matches=7, score_rows=14,
        attention_mask_shape_and_boundary=True), indent=2) + "\n", encoding="utf-8")
    print(comparison.round(4).to_string(index=False), flush=True)
    return summary, comparison


started = perf_counter()
parameters = smoke_test()
print(f"Transformer 검증 통과, 학습 파라미터 {parameters:,}개", flush=True)
train, _ = prepare_inputs()
evaluation = {match: validation_input(match, BASE) for match in MATCHES}
preparation_s = perf_counter() - started
fingerprint = {path.name: digest(path) for path in (NOTEBOOK, tuning, shared)}
checkpoint = OUT / "checkpoint.json"
if checkpoint.is_file():
    saved = json.loads(checkpoint.read_text())
    assert saved["fingerprints"] == fingerprint
    results = saved["results"]
else:
    results = []
completed = {fold["match"] for fold in results}
for held in MATCHES:
    if held in completed:
        continue
    results.append(evaluate_fold(held, train, evaluation))
    pending = checkpoint.with_suffix(".tmp")
    pending.write_text(json.dumps(dict(fingerprints=fingerprint, results=results)), encoding="utf-8")
    pending.replace(checkpoint)
summary, comparison = write_reports(results, fingerprint, parameters, preparation_s)